# Global ⊕ local subspace analysis

**Run directory:** `/home/kv133/Alpha-Earth-Embeddings-Research/week5/runs/global_local_subspace_20261005_173516`

This notebook embeds the experiment report and all publication figures (no recompute required).

- Config: `../runs/global_local_subspace_20261005_173516/config.json`
- Artifacts: `../runs/global_local_subspace_20261005_173516/artifacts/`
- Report source: `../runs/global_local_subspace_20261005_173516/REPORT.md`


# Global ⊕ local subspace analysis — report

**Run:** `global_local_subspace_20261005_173516`
**Hypothesis under test:** `S_country ≈ S_global ⊕ S_local,country`
**Seeds:** [0, 1, 2, 3, 4] · **max_per_class/city:** 5000

## A. Main empirical findings

### Is cross-country task information concentrated in a low-dimensional subspace?
- Optimal PLS k ≤ 5 for **67%** of ordered country pairs (mean best_k=4.5).
- See `figures/01_transfer_vs_k.png` and `artifacts/pair_summary.csv`.

| pair | best k | best PLS F1 | lr64 | gain | PLS-2 |
|---|---:|---:|---:|---:|---:|
| JPN→IND | 10 | 0.585 | 0.577 | +0.008 | 0.554 |
| IND→JPN | 2 | 0.634 | 0.335 | +0.299 | 0.634 |
| IND→USA | 1 | 0.759 | 0.427 | +0.332 | 0.728 |
| JPN→USA | 5 | 0.798 | 0.731 | +0.067 | 0.788 |
| USA→JPN | 8 | 0.694 | 0.630 | +0.063 | 0.616 |
| USA→IND | 1 | 0.554 | 0.362 | +0.192 | 0.527 |

### Are early task-relevant subspaces more similar across countries?
- Chordal/Grassmann distances generally **increase** with k (figure `04_subspace_distance_vs_k.png`).
- Label-shuffled PLS subspaces at k=2 are **more** distant than real ones (control: task structure needed for alignment).

### Do later dimensions become more geography-specific?
- Compare within-country / LORO curves vs cross-country OOD in `02_within_vs_cross_vs_k.png` and marginal tradeoff scatter `09_marginal_component_tradeoff.png`.
- **USA:** within ΔF1(k1→k20)=+0.048; cross ΔF1=-0.099.
- **JPN:** within ΔF1(k1→k20)=+0.065; cross ΔF1=-0.003.
- **IND:** within ΔF1(k1→k20)=+0.069; cross ΔF1=-0.257.

### Does subspace distance predict transfer degradation?

| distance | vs | Pearson | Spearman | 95% CI (Pearson) | n |
|---|---|---:|---:|---|---:|
| chordal_distance | ood_macro_f1 | -0.340 | -0.335 | [-0.405, -0.277] | 600 |
| chordal_distance | transfer_drop_f1 | 0.393 | 0.458 | [0.329, 0.459] | 600 |
| grassmann_geodesic | ood_macro_f1 | -0.339 | -0.333 | [-0.403, -0.277] | 600 |
| grassmann_geodesic | transfer_drop_f1 | 0.394 | 0.455 | [0.332, 0.459] | 600 |
| mean_principal_angle_deg | ood_macro_f1 | 0.123 | 0.158 | [0.040, 0.195] | 600 |
| mean_principal_angle_deg | transfer_drop_f1 | -0.165 | -0.257 | [-0.248, -0.072] | 600 |
| max_principal_angle_deg | ood_macro_f1 | -0.179 | -0.048 | [-0.238, -0.120] | 600 |
| max_principal_angle_deg | transfer_drop_f1 | 0.232 | 0.185 | [0.170, 0.296] | 600 |

- Chordal vs transfer_drop Pearson ≈ **0.393** (positive association as hypothesized).
- **Caveat:** only 3 countries ⇒ few unique geometries; treat correlations as exploratory.

### Does a consensus/global subspace improve unseen-country performance?

| method | mean OOD F1 | std |
|---|---:|---:|
| lr64 | 0.510 | 0.148 |
| pca | 0.530 | 0.146 |
| pls | 0.641 | 0.093 |
| pooled_pls | 0.597 | 0.172 |
| consensus_avgproj | 0.676 | 0.111 |
| consensus_karcher | 0.669 | 0.118 |
| oracle_target_pls | 0.729 | 0.079 |

Oracle target-country PLS is an upper bound only.

## B. Evidence for/against the hypothesis

**Overall: `partially supports`** for `S_country ≈ S_global ⊕ S_local,country`.

| Claim | Evidence |
|---|---|
| Low-dimensional transferable task structure | supports (early optimal k) |
| Early subspaces more aligned than late | supports |
| Distance predicts transfer drop | supports |
| Alignment requires true labels | supports |
| Local residual adds source-specific signal | see `06_shared_vs_local.png` |

## C. Alternative explanations

- **PLS as regularizer / anti-overfitting:** random projections and PCA are included; if PLS uniquely helps, not pure rank reduction.
- **Sample-size imbalance:** capped `max_per_class` per city; multi-seed.
- **Class imbalance:** analysis extract is 1:1 per city.
- **Few countries:** n=3 geographies — geometry–transfer regressions are underpowered.
- **Urban morphology / label noise:** not directly measured; flagged as open.
- **Component instability:** cumulative subspaces + Procrustes/principal angles used; raw component matching avoided.

## D. Paper-worthy claims

### Directly supported
1. Short supervised bottlenecks (often PLS k≈2–5) can outperform 64-D LR on several cross-country transfers.
2. Optimal bottleneck width is pair-dependent; India-involved transfers especially favor low rank.
3. Label-shuffled controls destroy meaningful early-subspace alignment (when observed).

### Promising hypotheses
1. Task-relevant AlphaEarth geometry ≈ shared low-d subspace + geography-specific residual.
2. Subspace distance metrics may track transfer degradation (needs more countries).

### Not yet publication-grade as CVPR claims
1. Universal global RES/NRES subspace across all geographies.
2. Causal claim that later PLS dims *are* geography IDs.
3. Any single k that is globally optimal.


## Figures


### 1. Cross-country transfer vs PLS/PCA/random dimensionality

![1. Cross-country transfer vs PLS/PCA/random dimensionality](../runs/global_local_subspace_20261005_173516/figures/01_transfer_vs_k.png)

`01_transfer_vs_k.png`


### 2. In-domain vs LORO vs cross-country as PLS rank grows

![2. In-domain vs LORO vs cross-country as PLS rank grows](../runs/global_local_subspace_20261005_173516/figures/02_within_vs_cross_vs_k.png)

`02_within_vs_cross_vs_k.png`


### 3. Principal-angle spectra at k=5

![3. Principal-angle spectra at k=5](../runs/global_local_subspace_20261005_173516/figures/03_principal_angle_spectra_k5.png)

`03_principal_angle_spectra_k5.png`


### 4. Chordal / Grassmann distance vs k

![4. Chordal / Grassmann distance vs k](../runs/global_local_subspace_20261005_173516/figures/04_subspace_distance_vs_k.png)

`04_subspace_distance_vs_k.png`


### 5. Subspace distance vs transfer performance & degradation

![5. Subspace distance vs transfer performance & degradation](../runs/global_local_subspace_20261005_173516/figures/05_distance_vs_transfer.png)

`05_distance_vs_transfer.png`


### 6. Global vs local residual contributions

![6. Global vs local residual contributions](../runs/global_local_subspace_20261005_173516/figures/06_shared_vs_local.png)

`06_shared_vs_local.png`


### 7. Summary: gain over lr64, optimal k, geometry vs PLS-2

![7. Summary: gain over lr64, optimal k, geometry vs PLS-2](../runs/global_local_subspace_20261005_173516/figures/07_summary_panels.png)

`07_summary_panels.png`


### 8. Shared/global subspace methods vs baselines

![8. Shared/global subspace methods vs baselines](../runs/global_local_subspace_20261005_173516/figures/08_consensus_methods.png)

`08_consensus_methods.png`


### 9. Marginal PLS dimensions: in-domain vs OOD ΔF1

![9. Marginal PLS dimensions: in-domain vs OOD ΔF1](../runs/global_local_subspace_20261005_173516/figures/09_marginal_component_tradeoff.png)

`09_marginal_component_tradeoff.png`


## Key tables (on disk)

- `../runs/global_local_subspace_20261005_173516/artifacts/pair_summary.csv`
- `../runs/global_local_subspace_20261005_173516/artifacts/transfer_k_sweep.csv`
- `../runs/global_local_subspace_20261005_173516/artifacts/geometry_transfer_correlations.csv`
- `../runs/global_local_subspace_20261005_173516/artifacts/global_consensus_transfer.csv`
- `../runs/global_local_subspace_20261005_173516/artifacts/shared_vs_local.csv`


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

RUN = Path(r"/home/kv133/Alpha-Earth-Embeddings-Research/week5/runs/global_local_subspace_20261005_173516")
display(Markdown("### pair_summary"))
display(pd.read_csv(RUN / "artifacts" / "pair_summary.csv"))
display(Markdown("### geometry ↔ transfer correlations"))
display(pd.read_csv(RUN / "artifacts" / "geometry_transfer_correlations.csv"))


### pair_summary

,pair,best_k,best_pls_f1,lr64_f1,gain_over_lr64,chordal_at_best_k,pls2_f1
0,JPN→IND,10,0.584747,0.577227,0.007520,2.442736,0.553875
1,IND→JPN,2,0.633544,0.334829,0.298714,1.343779,0.633544
2,IND→USA,1,0.759102,0.426749,0.332353,0.914139,0.727945
3,JPN→USA,5,0.797897,0.731287,0.066610,1.940672,0.788221
4,USA→JPN,8,0.693526,0.630069,0.063457,2.317676,0.616442
5,USA→IND,1,0.554355,0.362146,0.192209,0.914139,0.527460


### geometry ↔ transfer correlations

,distance_metric,performance_metric,pearson,spearman,pearson_ci95_lo,pearson_ci95_hi,n
0,chordal_distance,ood_macro_f1,-0.339846,-0.334839,-0.405130,-0.276588,600
1,chordal_distance,transfer_drop_f1,0.393475,0.457761,0.329282,0.459109,600
2,grassmann_geodesic,ood_macro_f1,-0.338645,-0.333143,-0.402659,-0.276723,600
3,grassmann_geodesic,transfer_drop_f1,0.393795,0.455379,0.331745,0.459290,600
4,mean_principal_angle_deg,ood_macro_f1,0.122606,0.158309,0.040340,0.195146,600
5,mean_principal_angle_deg,transfer_drop_f1,-0.164502,-0.256561,-0.247911,-0.071514,600
6,max_principal_angle_deg,ood_macro_f1,-0.178970,-0.047905,-0.238102,-0.119805,600
7,max_principal_angle_deg,transfer_drop_f1,0.231812,0.184571,0.169769,0.295896,600
